# Statistical Hypothesis Testing

## Step 1: Load Dataset

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../datasets/student_exam_features.csv')
print(f"Dataset: {df.shape}")
df.head()

Dataset: (10000, 35)


,student_id,gender,age,parental_education,family_income,internet_access,study_environment,study_hours_per_day,attendance_rate,sleep_hours,...,grade_A,grade_B,grade_C,grade_D,grade_F,productive_hours,study_attendance,engagement_score,disadvantage_index,social_study_ratio
0,S00001,Male,17,High School,Medium,Yes,Quiet,2.98,96.5,6.05,...,0,0,0,0,1,9.03,287.570,88.50,0,0.033557
1,S00002,Female,18,High School,Low,Yes,Quiet,4.45,95.7,6.96,...,0,0,1,0,0,11.41,425.865,83.30,1,0.651685
2,S00003,Male,17,High School,Medium,No,Quiet,3.75,76.0,7.02,...,0,0,0,0,1,10.77,285.000,76.80,1,0.640000
3,S00004,Male,18,Bachelor,Medium,Yes,Quiet,2.03,72.6,6.23,...,0,0,0,0,1,8.26,147.378,68.05,1,1.724138
4,S00005,Male,18,Bachelor,Medium,Yes,Quiet,5.14,87.3,8.54,...,0,0,1,0,0,13.68,448.722,79.55,1,0.408560


## Step 2: H1 — Pearson Correlations (Behavioral Features vs final_exam_score)

In [2]:
# Pearson correlation for key behavioral features vs final_exam_score
r_study, p_study = stats.pearsonr(df['study_hours_per_day'], df['final_exam_score'])
r_social, p_social = stats.pearsonr(df['social_media_hours'], df['final_exam_score'])
r_attend, p_attend = stats.pearsonr(df['attendance_rate'], df['final_exam_score'])
r_sleep, p_sleep = stats.pearsonr(df['sleep_hours'], df['final_exam_score'])
r_assign, p_assign = stats.pearsonr(df['assignment_completion_rate'], df['final_exam_score'])

print("H1 — Pearson correlations with final_exam_score:")
print(f"  study_hours_per_day:        r={r_study:.4f}, p={p_study:.2e}")
print(f"  social_media_hours:         r={r_social:.4f}, p={p_social:.2e}")
print(f"  attendance_rate:            r={r_attend:.4f}, p={p_attend:.2e}")
print(f"  sleep_hours:                r={r_sleep:.4f}, p={p_sleep:.2e}")
print(f"  assignment_completion_rate: r={r_assign:.4f}, p={p_assign:.2e}")

print(f"\nH1 Verification:")
print(f"  study_hours_per_day berkorelasi POSITIF: {'Ya' if r_study > 0 else 'Tidak'} (r={r_study:.4f}, p={p_study:.2e})")
print(f"  social_media_hours berkorelasi NEGATIF: {'Ya' if r_social < 0 else 'Tidak'} (r={r_social:.4f}, p={p_social:.2e})")

# Rank all behavioral features by absolute correlation
behavioral_features = ['study_hours_per_day', 'attendance_rate', 'sleep_hours',
                       'social_media_hours', 'assignment_completion_rate',
                       'online_courses_completed', 'previous_gpa']
correlations = {f: stats.pearsonr(df[f], df['final_exam_score'])[0] for f in behavioral_features}
ranked = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)
print(f"\n  Ranked behavioral predictors:")
for feat, r in ranked:
    print(f"    {feat}: r={r:.4f}")

H1 — Pearson correlations with final_exam_score:
  study_hours_per_day:        r=0.5758, p=0.00e+00
  social_media_hours:         r=-0.2463, p=4.54e-138
  attendance_rate:            r=0.1506, p=8.01e-52
  sleep_hours:                r=0.0279, p=5.31e-03
  assignment_completion_rate: r=0.1707, p=2.92e-66

H1 Verification:
  study_hours_per_day berkorelasi POSITIF: Ya (r=0.5758, p=0.00e+00)
  social_media_hours berkorelasi NEGATIF: Ya (r=-0.2463, p=4.54e-138)

  Ranked behavioral predictors:
    previous_gpa: r=0.8912
    study_hours_per_day: r=0.5758
    social_media_hours: r=-0.2463
    assignment_completion_rate: r=0.1707
    attendance_rate: r=0.1506
    sleep_hours: r=0.0279
    online_courses_completed: r=-0.0185


## Step 3: Supporting Tests — Tutoring & Internet Access (t-tests)

In [3]:
print("--- Supporting Tests for RQ1 ---")

# Tutoring impact
tutor_yes = df[df['tutoring'] == 'Yes']['final_exam_score']
tutor_no = df[df['tutoring'] == 'No']['final_exam_score']
t_stat, p_val = stats.ttest_ind(tutor_yes, tutor_no)
print(f"Tutoring impact: t={t_stat:.4f}, p={p_val:.2e}")
print(f"   Mean (Yes): {tutor_yes.mean():.2f}, Mean (No): {tutor_no.mean():.2f}")

# Internet access impact
inet_yes = df[df['internet_access'] == 'Yes']['final_exam_score']
inet_no = df[df['internet_access'] == 'No']['final_exam_score']
t_stat2, p_val2 = stats.ttest_ind(inet_yes, inet_no)
print(f"Internet access impact: t={t_stat2:.4f}, p={p_val2:.2e}")
print(f"   Mean (Yes): {inet_yes.mean():.2f}, Mean (No): {inet_no.mean():.2f}")

--- Supporting Tests for RQ1 ---
Tutoring impact: t=-0.8175, p=4.14e-01
   Mean (Yes): 49.53, Mean (No): 49.75
Internet access impact: t=2.1167, p=3.43e-02
   Mean (Yes): 49.77, Mean (No): 48.92


## Step 4: ANOVA — Parental Education vs Final Exam Score

In [4]:
groups = [group['final_exam_score'].values for name, group in df.groupby('parental_education')]
f_stat, p_val = stats.f_oneway(*groups)
print(f"ANOVA — Final exam score by parental education:")
print(f"  F-statistic: {f_stat:.4f}, p-value: {p_val:.2e}")

for name, group in df.groupby('parental_education'):
    print(f"    {name}: mean={group['final_exam_score'].mean():.2f} (n={len(group)})")

ANOVA — Final exam score by parental education:
  F-statistic: 1.0109, p-value: 3.87e-01
    Bachelor: mean=49.90 (n=3502)
    High School: mean=49.42 (n=3926)
    Master: mean=49.78 (n=2047)
    Phd: mean=49.79 (n=525)


## Step 5: ANOVA — Grade Category vs Study Habits

In [5]:
print("ANOVA — Study habits by grade category:")
for col in ['study_hours_per_day', 'sleep_hours', 'social_media_hours', 'attendance_rate']:
    groups = [g[col].values for _, g in df.groupby('grade_category')]
    f_stat, p_val = stats.f_oneway(*groups)
    print(f"  {col}: F={f_stat:.4f}, p={p_val:.2e}")

ANOVA — Study habits by grade category:
  study_hours_per_day: F=903.0819, p=0.00e+00
  sleep_hours: F=3.2745, p=1.08e-02
  social_media_hours: F=113.5436, p=7.79e-95
  attendance_rate: F=45.3882, p=7.67e-38


## Step 6: Chi-Square — Pass/Fail vs Categorical Features

In [6]:
print("Chi-square — pass_fail vs categorical features:")
for col in ['gender', 'parental_education', 'family_income', 'internet_access', 'tutoring', 'study_environment']:
    contingency = pd.crosstab(df[col], df['pass_fail'])
    chi2, p, dof, expected = chi2_contingency(contingency)
    print(f"  {col}: chi2={chi2:.4f}, p={p:.2e}")

Chi-square — pass_fail vs categorical features:
  gender: chi2=0.4005, p=5.27e-01
  parental_education: chi2=5.0485, p=1.68e-01
  family_income: chi2=1.0757, p=5.84e-01
  internet_access: chi2=3.2269, p=7.24e-02
  tutoring: chi2=0.1530, p=6.96e-01
  study_environment: chi2=1.9275, p=3.81e-01


## Step 6b: Effect Sizes, Confidence Intervals & Multiple-Comparison Correction

Beyond test statistics and p-values, report **effect sizes** (Cohen's d for t-tests, eta-squared for ANOVA), **95% CI** for H1 correlations (Fisher z), and apply **Benjamini-Hochberg FDR** correction to the RQ1 family of tests. With N=10,000, statistical significance is easy to reach, so effect sizes are essential to judge *practical* significance.

In [7]:
# Step 6b: Effect Sizes, 95% CI, and Multiple-Comparison Correction
print("=" * 70)
print("EFFECT SIZES, 95% CI & MULTIPLE-COMPARISON CORRECTION")
print("=" * 70)

def cohens_d_ind(a, b):
    n1, n2 = len(a), len(b)
    s_pooled = np.sqrt(((n1 - 1) * a.var(ddof=1) + (n2 - 1) * b.var(ddof=1)) / (n1 + n2 - 2))
    return (a.mean() - b.mean()) / s_pooled

def eta_squared_oneway(groups):
    grand = np.concatenate(groups)
    gm = grand.mean()
    ss_between = sum(len(g) * (g.mean() - gm) ** 2 for g in groups)
    ss_total = ((grand - gm) ** 2).sum()
    return ss_between / ss_total

def pearson_ci(r, n, alpha=0.05):
    z, se = np.arctanh(r), 1 / np.sqrt(n - 3)
    zc = stats.norm.ppf(1 - alpha / 2)
    return np.tanh(z - zc * se), np.tanh(z + zc * se)

def benjamini_hochberg(pvals):
    p = np.asarray(pvals, float); m = len(p)
    order = np.argsort(p); ranked = p[order]
    adj = ranked * m / np.arange(1, m + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    out = np.empty(m); out[order] = np.clip(adj, 0, 1)
    return out

n = len(df)
fam_labels, fam_p = [], []

print("\nH1 Pearson correlations (r, 95% CI via Fisher z):")
for f in ['study_hours_per_day', 'social_media_hours', 'attendance_rate',
          'sleep_hours', 'assignment_completion_rate']:
    r, p = stats.pearsonr(df[f], df['final_exam_score'])
    lo, hi = pearson_ci(r, n)
    print(f"  {f}: r={r:.4f} [{lo:.4f}, {hi:.4f}], p={p:.2e}")
    fam_labels.append(f"corr:{f}"); fam_p.append(p)

print("\nSupporting t-tests (Cohen's d):")
for col, lab in [('tutoring', 'tutoring'), ('internet_access', 'internet')]:
    a = df[df[col] == 'Yes']['final_exam_score']
    b = df[df[col] == 'No']['final_exam_score']
    t, p = stats.ttest_ind(a, b)
    print(f"  {lab}: t={t:.4f}, p={p:.2e}, Cohen's d={cohens_d_ind(a, b):.4f}")
    fam_labels.append(f"ttest:{lab}"); fam_p.append(p)

print("\nANOVA effect sizes (eta-squared):")
pe_groups = [g['final_exam_score'].values for _, g in df.groupby('parental_education')]
print(f"  parental_education: eta^2={eta_squared_oneway(pe_groups):.4f}")
for col in ['study_hours_per_day', 'social_media_hours', 'attendance_rate']:
    gg = [g[col].values for _, g in df.groupby('grade_category')]
    print(f"  grade x {col}: eta^2={eta_squared_oneway(gg):.4f}")

p_adj = benjamini_hochberg(fam_p)
print(f"\nBenjamini-Hochberg FDR correction (RQ1 family of {len(fam_p)} tests):")
for lab, p, pa in zip(fam_labels, fam_p, p_adj):
    print(f"  {lab}: p={p:.2e} -> p_adj={pa:.2e} ({'signifikan' if pa < 0.05 else 'TIDAK signifikan'})")

EFFECT SIZES, 95% CI & MULTIPLE-COMPARISON CORRECTION

H1 Pearson correlations (r, 95% CI via Fisher z):
  study_hours_per_day: r=0.5758 [0.5626, 0.5888], p=0.00e+00
  social_media_hours: r=-0.2463 [-0.2646, -0.2278], p=4.54e-138
  attendance_rate: r=0.1506 [0.1314, 0.1697], p=8.01e-52
  sleep_hours: r=0.0279 [0.0083, 0.0475], p=5.31e-03
  assignment_completion_rate: r=0.1707 [0.1516, 0.1897], p=2.92e-66

Supporting t-tests (Cohen's d):
  tutoring: t=-0.8175, p=4.14e-01, Cohen's d=-0.0178
  internet: t=2.1167, p=3.43e-02, Cohen's d=0.0701

ANOVA effect sizes (eta-squared):
  parental_education: eta^2=0.0003
  grade x study_hours_per_day: eta^2=0.2655
  grade x social_media_hours: eta^2=0.0435
  grade x attendance_rate: eta^2=0.0178

Benjamini-Hochberg FDR correction (RQ1 family of 7 tests):
  corr:study_hours_per_day: p=0.00e+00 -> p_adj=0.00e+00 (signifikan)
  corr:social_media_hours: p=4.54e-138 -> p_adj=1.59e-137 (signifikan)
  corr:attendance_rate: p=8.01e-52 -> p_adj=1.40e-51 (sig

## Step 7: Summary

In [8]:
print("=" * 60)
print("STATISTICAL TESTS SUMMARY")
print("=" * 60)
print(f"\nH1 — study_hours_per_day berkorelasi positif: {'Ya' if r_study > 0 else 'Tidak'} (r={r_study:.4f}, p={p_study:.2e})")
print(f"H1 — social_media_hours berkorelasi negatif: {'Ya' if r_social < 0 else 'Tidak'} (r={r_social:.4f}, p={p_social:.2e})")
h1_result = "DITERIMA" if (r_study > 0 and p_study < 0.05 and r_social < 0 and p_social < 0.05) else "DITOLAK"
print(f"\nH1 Status: {h1_result}")
print(f"\nNote: H2 (RF vs LR) dan H3 (clustering) akan diuji pada notebook terpisah.")

STATISTICAL TESTS SUMMARY

H1 — study_hours_per_day berkorelasi positif: Ya (r=0.5758, p=0.00e+00)
H1 — social_media_hours berkorelasi negatif: Ya (r=-0.2463, p=4.54e-138)

H1 Status: DITERIMA

Note: H2 (RF vs LR) dan H3 (clustering) akan diuji pada notebook terpisah.
